In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re

In [2]:
RESULTS_PREFIX = 'results'
HP_RESULTS_PREFIX = 'hpSearch'

In [3]:
input_dir = "/home-local2/sasah14.extra.nobkp/scratch/out/stta/sweep/"
sweep_names = ["PACS_resnet18-BN_ERM_0", "VLCS_resnet18-BN_ERM_0"]

In [4]:
def camel_case_split(identifier):
    matches = re.finditer('.+?(?:(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])|$)', identifier)
    return [m.group(0) for m in matches]

def get_test_env(args):
    test_envs = args['test_envs']
    assert len(test_envs) == 1
    test_env = test_envs[0]
    return test_env

def get_names(df_res, test_env):
    train_out_names = [col for col in df_res.columns
                            if 'env' in col and
                                'out_acc' in col and
                                str(test_env) not in col]
    test_out_names = [col for col in df_res.columns
                            if 'env' in col and
                                'out_acc' in col and
                                str(test_env) in col]
    test_in_names = [col for col in df_res.columns
                            if 'env' in col and
                                'in_acc' in col and
                                str(test_env) in col]
    assert len(test_out_names) == 1
    assert len(test_in_names) == 1
    test_out_name = test_out_names[0]
    test_in_name = test_in_names[0]
    return train_out_names, test_out_name, test_in_name

def fill_nans(acc_dict, test_in_name, adapt_algorithm, loc_alg_name):
    # Extract loc algo setting names
    find_algo, fix_algo, granularity = camel_case_split(loc_alg_name)

    # Append algo names to dictionary
    acc_dict['adapt_algorithm'].append(adapt_algorithm)
    acc_dict['find_algo'].append(find_algo)
    acc_dict['fix_algo'].append(fix_algo)
    acc_dict['granularity'].append(granularity)

    # Append target acc to dictionary
    acc_dict[test_in_name].append(np.nan)

def fill_dg(iid_dict, orc_dict, man_dict, df, train_out_names, test_out_name, test_in_name, args):
    # Get best train and test out idx
    df['train_out_mean'] = df[train_out_names].mean(axis=1)
    best_train_out_idx = df['train_out_mean'].idxmax()
    best_test_out_idx = df[test_out_name].idxmax()

    # Get best train and test out based test-in acc
    iidacc = df[test_in_name].iloc[best_train_out_idx]
    orcacc = df[test_in_name].iloc[best_test_out_idx]

    # TODO: Assumes that DG is encountered 1st, fix it!
    # Append acc to both dictionaries
    iid_dict[test_in_name] = [iidacc]
    orc_dict[test_in_name] = [orcacc]
    man_dict[test_in_name] = []
    
    # Append algo names to both dictionaries
    iid_dict['adapt_algorithm'] = ['DG']
    iid_dict['find_algo'] = ['DG']
    iid_dict['fix_algo'] = ['DG']
    iid_dict['granularity'] = ['DG']
    orc_dict['adapt_algorithm'] = ['DG']
    orc_dict['find_algo'] = ['DG']
    orc_dict['fix_algo'] = ['DG']
    orc_dict['granularity'] = ['DG']
    man_dict['adapt_algorithm'] = []
    man_dict['find_algo'] = []
    man_dict['fix_algo'] = []
    man_dict['granularity'] = []

    # Append seed to both dictionaries
    # iid_dict['hparams_seed'].append(args['hparams_seed'])
    # iid_dict['seed'].append(args['seed'])
    # iid_dict['trial_seed'].append(args['trial_seed'])
    # orc_dict['hparams_seed'].append(args['hparams_seed'])
    # orc_dict['seed'].append(args['seed'])
    # orc_dict['trial_seed'].append(args['trial_seed'])

def fill_row_loc(acc_dict, row, test_in_name, args):
    # Append algo names to dictionary
    acc_dict['adapt_algorithm'].append(args['adapt_algorithm'])
    if 'find_algo' in row: 
        # for loc TTA
        acc_dict['find_algo'].append(args['find_algo'])
        acc_dict['fix_algo'].append(args['fix_algo'])
        acc_dict['granularity'].append(args['granularity'])
    else:
        # for full TTA
        acc_dict['find_algo'].append('None')
        acc_dict['fix_algo'].append('None')
        acc_dict['granularity'].append('None')

    # Append target acc to dictionary
    acc_dict[test_in_name].append(row[test_in_name])

def fill_meta_data(acc_dict, test_in_name, dataset, backbone, dg_name, mark, hp_mode):
    acc_dict['hp_mode'] = [hp_mode for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['dataset'] = [dataset for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['backbone'] = [backbone for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['dg'] = [dg_name for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['mark'] = [mark for _ in range(len(acc_dict[test_in_name]))]

def get_df(acc_dict):
    # Create dataframe from dictionary
    df = pd.DataFrame.from_dict(acc_dict)

    # Compute average accuracy
    df['Avg'] = df[[col for col in df.columns
                            if 'in_acc' in col]].mean(axis=1)

    # Sort dataframe columns
    df = df[['adapt_algorithm', 'granularity', 'find_algo', 'fix_algo',
                     'env0_in_acc', 'env1_in_acc', 'env2_in_acc', 'env3_in_acc', 'Avg',
                     'hp_mode', 'dataset', 'backbone', 'dg', 'mark']]
    
    return df

In [5]:
sweep_tables = {}

for sweep_name in sweep_names:
    print(sweep_name)
    name_splits = sweep_name.split('_')
    dataset = name_splits[0]
    backbone = name_splits[1]
    dg_name = "_".join(name_splits[2:-1])
    mark = name_splits[-1]

    # Get sweep dir
    sweep_dir = os.path.join(input_dir, sweep_name)
    
    iidacc_dict, orcacc_dict = {}, {}
    manacc_dict = {}
    # iidacc_dict['hparams_seed'] = []
    # iidacc_dict['seed'] = []
    # iidacc_dict['trial_seed'] = []
    # orcacc_dict['hparams_seed'] = []
    # orcacc_dict['seed'] = []
    # orcacc_dict['trial_seed'] = []

    # Get runs
    runs = [run for run in sorted(os.listdir(sweep_dir))
                                            if os.path.isdir(os.path.join(sweep_dir, run))]

    # Get unique list of all results fnames
    res_fnames = sorted(list(set([fname for run in runs
                                            for fname in os.listdir(os.path.join(sweep_dir, run))
                                                if RESULTS_PREFIX in fname and\
                                                    'hp' not in fname and\
                                                    '.jsonl' in fname])))

    # Iterate over results fnames
    for run in tqdm(runs):
        run_to_testinname = {}  # dict to map a given run to test-in name
        for res_fname in res_fnames:
            # Get run dir and results path
            run_dir = os.path.join(sweep_dir, run)
            res_path = os.path.join(run_dir, res_fname)

            # check if results file exists
            if not os.path.exists(res_path):
                print('Warning: Results file not found for {}'.format(res_path))

                # Fill NaNs for all columns
                print(run_to_testinname)
                res_name_split = res_fname[:-6].split('_')
                fill_nans(iidacc_dict, run_to_testinname[run], res_name_split[1], res_name_split[2])
                fill_nans(orcacc_dict, run_to_testinname[run], res_name_split[1], res_name_split[2])
                fill_nans(manacc_dict, run_to_testinname[run], res_name_split[1], res_name_split[2])

            else:
                # print('Results file found for {}'.format(res_path))

                # Read in results
                df_res = pd.read_json(os.path.join(run_dir, res_fname), lines=True)

                # Get args
                args = df_res['args'][0]

                # Get test env
                test_env = get_test_env(args)

                # Get train and test in and out names
                train_out_names, test_out_name, test_in_name = get_names(df_res, test_env)

                res_name_split = res_fname[:-6].split('_')
                if len(res_name_split) == 1:
                    # Fill DG
                    fill_dg(iidacc_dict, orcacc_dict, manacc_dict, df_res, train_out_names, test_out_name, test_in_name, args)
                    run_to_testinname[run] = test_in_name

                else:
                    if len(res_name_split) != 3:
                        print('Warning: Processing unknown results file name: {}'.format(res_fname))
                    loc_iid_flag, loc_orc_flag, loc_man_flag = False, False, False
                    for idx, row in df_res.iterrows():
                        hp_mode = row['hp_mode']
                        if hp_mode == 'Iid':
                            fill_row_loc(iidacc_dict, row, test_in_name, args)
                            loc_iid_flag = True
                        
                        elif hp_mode == 'Orc':
                            fill_row_loc(orcacc_dict, row, test_in_name, args)
                            loc_orc_flag = True

                        elif hp_mode == 'Man':
                            fill_row_loc(manacc_dict, row, test_in_name, args)
                            loc_man_flag = True
                    
                    # Check if both orc and iid are called
                    if not loc_iid_flag:
                        print('Warning: No Iid results found for {}'.format(res_fname))
                        fill_nans(iidacc_dict, run_to_testinname[run], res_name_split[1], res_name_split[2])
                    if not loc_orc_flag:
                        print('Warning: No Orc results found for {}'.format(res_fname))
                        fill_nans(orcacc_dict, run_to_testinname[run], res_name_split[1], res_name_split[2])
                    if not loc_man_flag:
                        print('Warning: No Man results found for {}'.format(res_fname))
                        fill_nans(manacc_dict, run_to_testinname[run], res_name_split[1], res_name_split[2])


    # Check if dictionary is empty
    if len(iidacc_dict) == 0 or len(orcacc_dict) == 0 or len(manacc_dict) == 0:
        print('Warning: Empty dictionaries created for {}'.format(sweep_name))
        continue

    # Fill meta data
    fill_meta_data(iidacc_dict, test_in_name, dataset, backbone, dg_name, mark, 'Iid')
    fill_meta_data(orcacc_dict, test_in_name, dataset, backbone, dg_name, mark, 'Orc')
    fill_meta_data(manacc_dict, test_in_name, dataset, backbone, dg_name, mark, 'Man')

    # Create dataframe from dictionary
    df_iid = get_df(iidacc_dict)
    df_orc = get_df(orcacc_dict)
    df_man = get_df(manacc_dict)

    # Save dataframe to csv
    df_iid.to_csv(os.path.join(sweep_dir, sweep_name + '_iid.csv'), index=False)
    df_orc.to_csv(os.path.join(sweep_dir, sweep_name + '_orc.csv'), index=False)
    df_man.to_csv(os.path.join(sweep_dir, sweep_name + '_man.csv'), index=False)

    # Add dataframe to sweep tables dictionary
    sweep_tables[sweep_name + '_iid'] = df_iid
    sweep_tables[sweep_name + '_orc'] = df_orc
    sweep_tables[sweep_name + '_man'] = df_man

# Check if sweep_tables is empty or contains single dataframe
if len(sweep_tables) == 0:
    print('Warning: Empty sweep_tables created! No results found!')
elif len(sweep_tables) == 2:
    print('Only single dataframe found in sweep_tables! No merging needed!')

    # Save merged dataframes to csv
    df_iid.to_csv(os.path.join(input_dir, 'iid.csv'), index=False)
    df_orc.to_csv(os.path.join(input_dir, 'orc.csv'), index=False)
    df_man.to_csv(os.path.join(input_dir, 'man.csv'), index=False)
else:
    print('Merging dataframes...')
    # Merge all dataframes in sweep_tables
    df_all_iid = pd.concat([df for dfname, df in sweep_tables.items() if 'iid' in dfname])
    df_all_orc = pd.concat([df for dfname, df in sweep_tables.items() if 'orc' in dfname])
    df_all_man = pd.concat([df for dfname, df in sweep_tables.items() if 'man' in dfname])

    # Save merged dataframes to csv
    df_all_iid.to_csv(os.path.join(input_dir, 'iid.csv'), index=False)
    df_all_orc.to_csv(os.path.join(input_dir, 'orc.csv'), index=False)
    df_all_man.to_csv(os.path.join(input_dir, 'man.csv'), index=False)

PACS_resnet18-BN_ERM_0


 25%|██▌       | 1/4 [00:00<00:00,  7.41it/s]

{'075e460d9087c1c387f1d20ef5962584': 'env1_in_acc'}
{'1a9a390e4994c0e50882ef9931667273': 'env2_in_acc'}


100%|██████████| 4/4 [00:00<00:00,  8.60it/s]


{'8fa18032b0cfb71692121bef458c6b74': 'env0_in_acc'}
{'f5ce2006559d010d17f088e5cc685b62': 'env3_in_acc'}
VLCS_resnet18-BN_ERM_0


100%|██████████| 4/4 [00:00<00:00, 10.47it/s]


Merging dataframes...


In [6]:
# Get sweep dir
sweep_dir = os.path.join(input_dir, sweep_name)

# Get runs
runs = [run for run in sorted(os.listdir(sweep_dir))
                                        if os.path.isdir(os.path.join(sweep_dir, run))]

# Get unique list of all results fnames
res_fnames = sorted(list(set([fname for run in runs
                                        for fname in os.listdir(os.path.join(sweep_dir, run))
                                            if RESULTS_PREFIX in fname and\
                                                'hp' not in fname and\
                                                '.jsonl' in fname])))

# Iterate over results fnames
run = runs[0]
res_fname = res_fnames[0]

# Get run dir and results path
run_dir = os.path.join(sweep_dir, run)
res_path = os.path.join(run_dir, res_fname)

print('Results file {}'.format(res_path))

# Read in results
df_res = pd.read_json(os.path.join(run_dir, res_fname), lines=True)

# Get args
args = df_res['args'][0]

# Get test env
test_env = get_test_env(args)

# Get train and test in and out names
train_out_names, test_out_name, test_in_name = get_names(df_res, test_env)

Results file /home-local2/sasah14.extra.nobkp/scratch/out/stta/sweep/VLCS_resnet18-BN_ERM_0/27d7a532761f106584b5338d28998333/results.jsonl
